# Self-Life Estimation Regression

Trains SVR, GB, and GP regressors for shelf-life estimation from
S11 spectra under two cross-validation schemes (LOGO, K-Fold), then derives a binary
fresh/not-fresh classifier from the regressors.

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import LeaveOneGroupOut, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import WhiteKernel, RBF
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    'font.size': 10,
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# File paths
FILE_PATH    = 'processed_data.pkl'
PRELIM_PATH  = 'preliminary_results.pkl'
RESULTS_PATH = 'regression_results_v5.pkl'
HYPERPARAMS_PATH = {
    'LOGO':  'hyperparams_v5_logo.pkl',
    'KFold': 'hyperparams_v5_kfold.pkl',
}

# Constants
RANDOM_STATE    = 42
N_OPTUNA_TRIALS = 200
N_BOOTSTRAP     = 1000
KFOLD_N_SPLITS  = 5
np.random.seed(RANDOM_STATE)

# Execution control
# Set SKIP_TRAINING = True to load cached hyperparameters for both CV schemes
# and skip Optuna. Cache files are written at the end of Section 4.
SKIP_TRAINING = False

# CV scheme definitions
# Each entry: cv_name -> (cv_type, dataframe_group_column_or_None)
# cv_type = 'logo' uses LeaveOneGroupOut; 'kfold' uses KFold(KFOLD_N_SPLITS)
CV_SCHEMES = {
    'LOGO':  ('logo',  'target_class_date'),  # 5 folds, cross-batch
    'KFold': ('kfold', None),                 # 5 folds, random sweep-level split
    # 'LOGO-F':  ('logo',  'filename'),  # 5 folds, cross-batch
}

## Load Data & Configuration

In [ ]:
try:
    df = pd.read_pickle(FILE_PATH)
    
    print(f"Loaded '{FILE_PATH}': {len(df)} sweeps, "
          f"{df['target_class_date'].nunique()} best-before classes, "
          f"{df['filename'].nunique()} measurement epochs.")
except FileNotFoundError:
    raise FileNotFoundError(f"'{FILE_PATH}' not found. Run 02_loading_v2.ipynb first.")

try:
    with open(PRELIM_PATH, 'rb') as f:
        prelim = pickle.load(f)
    print(f"Loaded '{PRELIM_PATH}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"'{PRELIM_PATH}' not found. Run 02_preliminary_v2.ipynb first.")

# Manual overrides
OVERRIDE_REPR       = 'real_imag'  # None -> use per-sensor prelim recommendation
OVERRIDE_COMPONENTS = 10           # None -> use per-sensor prelim recommendation
OVERRIDE_NOISE      = 0.0025       # None -> use empirical noise level from prelim

NOISE_LEVEL = OVERRIDE_NOISE if OVERRIDE_NOISE is not None else prelim['noise_level']

def _resolve(override, prelim_val):
    return override if override is not None else prelim_val

SENSOR_CONFIG = {
    'Sensor A \u2014 GO/Nafion (S11)': {
        'feature_col':  'features_a',
        'best_repr':    _resolve(OVERRIDE_REPR,       prelim['sensor_a']['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim['sensor_a']['n_components']),
    },
    'Sensor B \u2014 G/GO/PEDOT:PSS (S22)': {
        'feature_col':  'features_b',
        'best_repr':    _resolve(OVERRIDE_REPR,       prelim['sensor_b']['best_repr']),
        'n_components': _resolve(OVERRIDE_COMPONENTS, prelim['sensor_b']['n_components']),
    },
}

for sname, cfg in SENSOR_CONFIG.items():
    print(f"  {sname}: repr='{cfg['best_repr']}'  n_comp={cfg['n_components']}")
print(f"  Noise level: {NOISE_LEVEL}")

## Feature Representation Builder

In [ ]:
def build_representation(X_raw: np.ndarray, repr_name: str) -> np.ndarray:
    """Converts a raw [Real | Imag] feature matrix to the specified representation."""
    n = X_raw.shape[1] // 2
    real, imag = X_raw[:, :n], X_raw[:, n:]
    if repr_name == 'real_imag':
        return X_raw
    elif repr_name == 'magnitude':
        return np.sqrt(real**2 + imag**2)
    elif repr_name == 'phase':
        return np.arctan2(imag, real)
    else:
        raise ValueError(f"Unknown representation: '{repr_name}'")

## Helper Functions

In [ ]:
def precompute_inner_logo_folds(X_train, y_train, groups_train, n_components):
    """
    Pre-bakes augmentation, scaling, and PCA for each LOGO inner fold.
    Returns a list of (Xtr_pca, ytr, Xte_pca, yte) tuples.
    """
    logo = LeaveOneGroupOut()
    folded_data = []
    for tr_idx, te_idx in logo.split(X_train, y_train, groups_train):
        Xtr, Xte = X_train[tr_idx], X_train[te_idx]
        ytr, yte = y_train[tr_idx], y_train[te_idx]
        Xtr = Xtr + np.random.normal(0, NOISE_LEVEL, Xtr.shape)
        scaler = StandardScaler()
        Xtr_sc = scaler.fit_transform(Xtr)
        Xte_sc = scaler.transform(Xte)
        pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
        Xtr_pca = pca.fit_transform(Xtr_sc)
        Xte_pca = pca.transform(Xte_sc)
        folded_data.append((Xtr_pca, ytr, Xte_pca, yte))
    return folded_data


def precompute_inner_kfold_folds(X_train, y_train, n_splits, n_components):
    """
    Pre-bakes augmentation, scaling, and PCA for each K-Fold inner split.
    Returns a list of (Xtr_pca, ytr, Xte_pca, yte) tuples.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    folded_data = []
    for tr_idx, te_idx in kf.split(X_train):
        Xtr, Xte = X_train[tr_idx], X_train[te_idx]
        ytr, yte = y_train[tr_idx], y_train[te_idx]
        Xtr = Xtr + np.random.normal(0, NOISE_LEVEL, Xtr.shape)
        scaler = StandardScaler()
        Xtr_sc = scaler.fit_transform(Xtr)
        Xte_sc = scaler.transform(Xte)
        pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
        Xtr_pca = pca.fit_transform(Xtr_sc)
        Xte_pca = pca.transform(Xte_sc)
        folded_data.append((Xtr_pca, ytr, Xte_pca, yte))
    return folded_data


def make_objective(model_name, precomputed_folds):
    """Returns an Optuna objective that scores a model on pre-baked fold data."""
    def objective(trial):
        if model_name == 'PCA + SVR':
            reg = SVR(
                kernel='rbf', max_iter=50000,
                C=trial.suggest_float('C', 1e-1, 1e3, log=True),
                epsilon=trial.suggest_float('epsilon', 1e-3, 1.0, log=True),
                gamma=trial.suggest_categorical('gamma', ['scale', 'auto']),
            )
        elif model_name == 'PCA + GB':
            reg = GradientBoostingRegressor(
                random_state=RANDOM_STATE,
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                max_depth=trial.suggest_int('max_depth', 2, 6),
                subsample=trial.suggest_float('subsample', 0.5, 1.0),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
            )
        else:
            raise ValueError(f"Unexpected model in Optuna objective: {model_name}")
        fold_mae = []
        for Xtr_pca, ytr, Xte_pca, yte in precomputed_folds:
            reg.fit(Xtr_pca, ytr)
            fold_mae.append(mean_absolute_error(yte, reg.predict(Xte_pca)))
        return float(np.mean(fold_mae))
    return objective


def optimise_hyperparams(model_name, precomputed_folds, n_trials):
    """Runs Optuna TPE on pre-baked folds. Returns (best_params, best_mae)."""
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(
        make_objective(model_name, precomputed_folds),
        n_trials=n_trials,
        show_progress_bar=True,
        n_jobs=-1,
    )
    return study.best_params, study.best_value


def build_tuned_pipeline(model_name, best_params, n_components):
    """Constructs a full sklearn Pipeline from tuned hyperparameters."""
    if model_name == 'PCA + SVR':
        reg = SVR(kernel='rbf', max_iter=50000,
                  C=best_params['C'],
                  epsilon=best_params['epsilon'],
                  gamma=best_params['gamma'])
    elif model_name == 'PCA + GB':
        reg = GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            n_estimators=best_params['n_estimators'],
            learning_rate=best_params['learning_rate'],
            max_depth=best_params['max_depth'],
            subsample=best_params['subsample'],
            min_samples_leaf=best_params['min_samples_leaf'],
        )
    elif model_name == 'PCA + GP':
        reg = GaussianProcessRegressor(
            kernel=1.0 * RBF(length_scale=1.0,
                              length_scale_bounds=(1e-2, 1e3))
                   + WhiteKernel(noise_level=0.1,
                                 noise_level_bounds=(1e-5, 1e2)),
            normalize_y=True,
            n_restarts_optimizer=3,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=n_components, random_state=RANDOM_STATE)),
        ('reg',    reg),
    ])


def bootstrap_ci(y_true, y_pred, n_bootstrap=N_BOOTSTRAP, ci=0.95):
    """Percentile bootstrap CIs on MAE and R². Returns dict with 'mae' and 'r2'."""
    rng = np.random.default_rng(RANDOM_STATE)
    n = len(y_true)
    mae_b, r2_b = [], []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        mae_b.append(mean_absolute_error(y_true[idx], y_pred[idx]))
        r2_b.append(r2_score(y_true[idx], y_pred[idx]))
    alpha = (1 - ci) / 2
    return {
        'mae': (np.mean(mae_b),
                np.percentile(mae_b, alpha * 100),
                np.percentile(mae_b, (1 - alpha) * 100)),
        'r2':  (np.mean(r2_b),
                np.percentile(r2_b,  alpha * 100),
                np.percentile(r2_b,  (1 - alpha) * 100)),
    }


def binary_classification_metrics(y_true, y_pred):
    """
    Thresholds regression predictions at 0 to derive fresh/spoiled labels.
    Uses continuous prediction as the ranking score for ROC-AUC.
    fresh = 1 (y > 0, before best-before), spoiled = 0 (y <= 0, past best-before).
    """
    y_true_bin = (y_true > 0).astype(int)
    y_pred_bin = (y_pred > 0).astype(int)
    return {
        'accuracy':  accuracy_score(y_true_bin, y_pred_bin),
        'precision': precision_score(y_true_bin, y_pred_bin, zero_division=0),
        'recall':    recall_score(y_true_bin, y_pred_bin, zero_division=0),
        'f1':        f1_score(y_true_bin, y_pred_bin, zero_division=0),
        'roc_auc':   roc_auc_score(y_true_bin, y_pred),
        'y_true_bin': y_true_bin,
        'y_score':    y_pred,
    }

## Main Training and Evaluation Loop

For each CV scheme and sensor precomputes the inner folds and runs Optuna for
SVR and GB once, then runs the outer CV loop with the computed hyperparameters,
and finally thresholds the predictions at y = 0 (binary classfier).

In [ ]:
MODEL_NAMES = ['PCA + SVR', 'PCA + GB', 'PCA + GP']
MODEL_NAMES = ['PCA + SVR', 'PCA + GB']

all_cv_results     = {}   # {cv_name: {sensor_name: results_dict}}
all_cv_hyperparams = {}   # {cv_name: {sensor_name: best_params}}

for cv_name, (cv_type, group_col) in CV_SCHEMES.items():
    print(f"\n{'#'*70}")
    print(f"  CV SCHEME: {cv_name}  ({'groups=' + group_col if group_col else 'random K=' + str(KFOLD_N_SPLITS)})")
    print(f"{'#'*70}")

    all_logo_results = {}
    all_hyperparams  = {}

    # Load cached hyperparameters if requested
    cached_hyperparams = {}
    if SKIP_TRAINING:
        try:
            with open(HYPERPARAMS_PATH[cv_name], 'rb') as f:
                cached_hyperparams = pickle.load(f)
            print(f"Loaded cached hyperparameters from '{HYPERPARAMS_PATH[cv_name]}'.")
            print("Optuna search will be skipped.\n")
        except FileNotFoundError:
            print(f"WARNING: '{HYPERPARAMS_PATH[cv_name]}' not found. Running full Optuna search.\n")

    for sensor_name, cfg in SENSOR_CONFIG.items():
        print(f"\n{'='*70}")
        print(f"  SENSOR: {sensor_name}")
        print(f"  repr='{cfg['best_repr']}'  N_COMPONENTS={cfg['n_components']}")
        print(f"{'='*70}")

        X_raw  = np.stack(df[cfg['feature_col']].values)
        X      = build_representation(X_raw, cfg['best_repr'])
        y      = df['timedelta_days'].values
        groups = df[group_col].values if group_col else None
        n_comp = cfg['n_components']

        if cv_type == 'logo':
            n_folds = len(np.unique(groups))
        else:
            n_folds = KFOLD_N_SPLITS

        # Phase 1: Tune hyperparameters once
        best_params  = {}
        tuning_folds = None

        for model_name in MODEL_NAMES:
            if model_name == 'PCA + GP':
                best_params[model_name] = None
                print(f"  {model_name:12s}: no Optuna (internal kernel optimisation)")
                continue

            cached = cached_hyperparams.get(sensor_name, {}).get(model_name, 'MISSING')
            if cached != 'MISSING' and cached is not None:
                best_params[model_name] = cached
                print(f"  {model_name:12s}: [cached]  {cached}")
            else:
                if tuning_folds is None:
                    if cv_type == 'logo':
                        print(f"  Precomputing {n_folds}-fold LOGO for Optuna...")
                        tuning_folds = precompute_inner_logo_folds(X, y, groups, n_comp)
                    else:
                        print(f"  Precomputing {n_folds}-fold K-Fold for Optuna...")
                        tuning_folds = precompute_inner_kfold_folds(X, y, KFOLD_N_SPLITS, n_comp)
                bp, bv = optimise_hyperparams(model_name, tuning_folds, N_OPTUNA_TRIALS)
                best_params[model_name] = bp
                print(f"  {model_name:12s}: {cv_name}-MAE={bv:.4f}  {bp}")

        # Phase 2: CV evaluation with fixed hyperparameters
        predictions   = {name: {'y_true': [], 'y_pred': []} for name in MODEL_NAMES}
        fold_metrics  = {name: {'mae': [], 'r2': []}         for name in MODEL_NAMES}
        fold_held_out = []

        if cv_type == 'logo':
            split_iter = LeaveOneGroupOut().split(X, y, groups)
        else:
            split_iter = KFold(n_splits=KFOLD_N_SPLITS, shuffle=True,
                               random_state=RANDOM_STATE).split(X, y)

        print()
        for fold_i, (tr_idx, te_idx) in enumerate(split_iter):
            held_out = groups[te_idx[0]] if groups is not None else f'Fold {fold_i+1}'
            fold_held_out.append(held_out)

            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]
            X_tr_noisy = X_tr + np.random.normal(0, NOISE_LEVEL, X_tr.shape)

            fold_line = []
            for model_name in MODEL_NAMES:
                pipe   = build_tuned_pipeline(model_name, best_params[model_name], n_comp)
                pipe.fit(X_tr_noisy, y_tr)
                y_pred = pipe.predict(X_te)

                predictions[model_name]['y_true'].extend(y_te)
                predictions[model_name]['y_pred'].extend(y_pred)

                fm  = mean_absolute_error(y_te, y_pred)
                fr2 = r2_score(y_te, y_pred)
                fold_metrics[model_name]['mae'].append(fm)
                fold_metrics[model_name]['r2'].append(fr2)
                fold_line.append(f"{model_name.split('+')[1].strip()}: MAE={fm:.3f} R\u00b2={fr2:.3f}")

            print(f"  Fold {fold_i+1}/{n_folds} ({held_out}) \u2192 {'  |  '.join(fold_line)}")

        # Aggregate + binary classification
        logo_results = {}
        for model_name in MODEL_NAMES:
            y_true_all = np.array(predictions[model_name]['y_true'])
            y_pred_all = np.array(predictions[model_name]['y_pred'])
            ci         = bootstrap_ci(y_true_all, y_pred_all)
            pr, pp     = pearsonr(y_true_all, y_pred_all)
            sr, sp     = spearmanr(y_true_all, y_pred_all)
            bin_met    = binary_classification_metrics(y_true_all, y_pred_all)

            logo_results[model_name] = {
                'r2':            r2_score(y_true_all, y_pred_all),
                'mae':           mean_absolute_error(y_true_all, y_pred_all),
                'rmse':          np.sqrt(mean_squared_error(y_true_all, y_pred_all)),
                'pearson_r':     pr,  'pearson_p':  pp,
                'spearman_rho':  sr,  'spearman_p': sp,
                'ci_mae':        ci['mae'],
                'ci_r2':         ci['r2'],
                'fold_mae':      fold_metrics[model_name]['mae'],
                'fold_r2':       fold_metrics[model_name]['r2'],
                'fold_held_out': fold_held_out,
                'hyperparams':   best_params[model_name],
                'y_true':        y_true_all,
                'y_pred':        y_pred_all,
                'binary':        bin_met,
            }

        all_logo_results[sensor_name] = logo_results
        all_hyperparams[sensor_name]  = best_params

        print(f"\n{'\u2500'*70}")
        print(f"  {cv_name} RESULTS \u2014 {sensor_name}")
        for name, res in logo_results.items():
            ci_mae = res['ci_mae']
            ci_r2  = res['ci_r2']
            b      = res['binary']
            print(f"  {name:12s}  R\u00b2={res['r2']:.4f} [{ci_r2[1]:.4f},{ci_r2[2]:.4f}]  "
                  f"MAE={res['mae']:.4f}  "
                  f"\u03c1={res['spearman_rho']:.4f}  "
                  f"Acc={b['accuracy']:.4f}  AUC={b['roc_auc']:.4f}")

    all_cv_results[cv_name]     = all_logo_results
    all_cv_hyperparams[cv_name] = all_hyperparams

In [ ]:
# Save hyperparameters
for cv_name, hyperparams in all_cv_hyperparams.items():
    with open(HYPERPARAMS_PATH[cv_name], 'wb') as f:
        pickle.dump(hyperparams, f)
    print(f"Hyperparameters saved to '{HYPERPARAMS_PATH[cv_name]}'.")

print("\nSet SKIP_TRAINING = True to skip Optuna next run.")

## Visualisation

In [ ]:
# True vs Predicted scatter - one figure per CV scheme x sensor
for cv_name, logo_results_by_sensor in all_cv_results.items():
    for sensor_name, logo_res in logo_results_by_sensor.items():
        n_models = len(logo_res)
        fig, axes = plt.subplots(1, n_models, figsize=(5.5 * n_models, 5))
        if n_models == 1:
            axes = [axes]
        fig.suptitle(f'{cv_name}: True vs Predicted \u2014 {sensor_name}', fontsize=13)

        for ax, (name, res) in zip(axes, logo_res.items()):
            y_true, y_pred = res['y_true'], res['y_pred']
            ci_mae, ci_r2  = res['ci_mae'], res['ci_r2']

            sc = ax.scatter(y_true, y_pred, c=y_true, cmap='viridis',
                            alpha=0.4, s=25, edgecolors='none')
            lo = min(y_true.min(), y_pred.min())
            hi = max(y_true.max(), y_pred.max())
            ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Ideal')
            ax.axvline(0, color='grey', ls=':', lw=1, alpha=0.6)
            ax.axhline(0, color='grey', ls=':', lw=1, alpha=0.6)
            plt.colorbar(sc, ax=ax, label='Days')

            ax.set_title(
                f"{name}\n"
                f"R\u00b2={res['r2']:.3f} [{ci_r2[1]:.3f}\u2013{ci_r2[2]:.3f}]  "
                f"MAE={res['mae']:.2f} [{ci_mae[1]:.2f}\u2013{ci_mae[2]:.2f}] d  "
                f"\u03c1={res['spearman_rho']:.3f}",
                fontsize=9)
            ax.set_xlabel('True (days)')
            ax.set_ylabel('Predicted (days)')
            ax.grid(True, ls='--', alpha=0.4)

        plt.tight_layout()
        plt.show()

In [ ]:
# Per-fold MAE bar chart - one figure per CV scheme x sensor
for cv_name, logo_results_by_sensor in all_cv_results.items():
    for sensor_name, logo_res in logo_results_by_sensor.items():
        n_models  = len(logo_res)
        any_res   = next(iter(logo_res.values()))
        n_folds   = len(any_res['fold_mae'])
        fig_w     = max(5 * n_models, n_folds * 0.8 * n_models)
        fig, axes = plt.subplots(1, n_models, figsize=(fig_w, 4), sharey=False)
        if n_models == 1:
            axes = [axes]
        fig.suptitle(f'Per-Fold {cv_name} MAE \u2014 {sensor_name}', fontsize=12)

        for ax, (name, res) in zip(axes, logo_res.items()):
            fold_maes = res['fold_mae']
            held_out  = [str(g) for g in res['fold_held_out']]
            mean_mae  = np.mean(fold_maes)
            ci_mae    = res['ci_mae']

            ax.bar(range(1, len(fold_maes) + 1), fold_maes,
                   color='steelblue', edgecolor='black', alpha=0.8)
            ax.axhline(mean_mae, color='crimson', ls='--', lw=1.5,
                       label=f'Mean = {mean_mae:.2f} d')
            ax.axhline(ci_mae[1], color='grey', ls=':', lw=1)
            ax.axhline(ci_mae[2], color='grey', ls=':', lw=1,
                       label=f'95% CI [{ci_mae[1]:.2f}\u2013{ci_mae[2]:.2f}]')

            ax.set_xticks(range(1, len(fold_maes) + 1))
            ax.set_xticklabels(held_out, rotation=30, ha='right', fontsize=8)
            ax.set_title(name, fontsize=10)
            ax.set_xlabel('Held-out group / fold')
            ax.set_ylabel('MAE (days)')
            ax.legend(fontsize=8)
            ax.grid(axis='y', ls='--', alpha=0.4)

        plt.tight_layout()
        plt.show()

In [ ]:
# ROC curves - one figure per sensor, LOGO and K-Fold overlaid, one subplot per model
for sensor_name in SENSOR_CONFIG:
    n_models = len(MODEL_NAMES)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
    if n_models == 1:
        axes = [axes]
    fig.suptitle(f'ROC Curves (fresh vs. past best-before) \u2014 {sensor_name}', fontsize=12)

    colors = {'LOGO': 'steelblue', 'KFold': 'darkorange'}

    for ax, model_name in zip(axes, MODEL_NAMES):
        for cv_name in CV_SCHEMES:
            if sensor_name not in all_cv_results.get(cv_name, {}):
                continue
            res = all_cv_results[cv_name][sensor_name][model_name]
            b   = res['binary']
            fpr, tpr, _ = roc_curve(b['y_true_bin'], b['y_score'])
            ax.plot(fpr, tpr, color=colors[cv_name], lw=2,
                    label=f"{cv_name} AUC={b['roc_auc']:.3f}")

        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
        ax.set_title(model_name, fontsize=10)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.legend(fontsize=9)
        ax.grid(True, ls='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

## Summary Tables

In [ ]:
# Regression summary
reg_rows = []
for cv_name, logo_results_by_sensor in all_cv_results.items():
    for sensor_name, logo_res in logo_results_by_sensor.items():
        for model_name, res in logo_res.items():
            ci_mae = res['ci_mae']
            ci_r2  = res['ci_r2']
            reg_rows.append({
                'CV':             cv_name,
                'Sensor':         sensor_name,
                'Model':          model_name,
                'R\u00b2':             round(res['r2'],  4),
                'R\u00b2 95% CI':      f"[{ci_r2[1]:.4f}, {ci_r2[2]:.4f}]",
                'MAE (d)':        round(res['mae'], 4),
                'MAE 95% CI':     f"[{ci_mae[1]:.4f}, {ci_mae[2]:.4f}]",
                'RMSE (d)':       round(res['rmse'], 4),
                'Pearson r':      round(res['pearson_r'],    4),
                'Pearson p':      f"{res['pearson_p']:.2e}",
                'Spearman \u03c1':    round(res['spearman_rho'], 4),
                'Spearman p':     f"{res['spearman_p']:.2e}",
            })

df_reg = pd.DataFrame(reg_rows)
print("\n" + "="*20 + " REGRESSION RESULTS " + "="*20)
display(df_reg.sort_values(by=['CV', 'Sensor', 'MAE (d)']).round(4))

In [ ]:
# Binary classification summary
clf_rows = []
for cv_name, logo_results_by_sensor in all_cv_results.items():
    for sensor_name, logo_res in logo_results_by_sensor.items():
        for model_name, res in logo_res.items():
            b = res['binary']
            clf_rows.append({
                'CV':        cv_name,
                'Sensor':    sensor_name,
                'Model':     model_name,
                'Accuracy':  round(b['accuracy'],  4),
                'Precision': round(b['precision'], 4),
                'Recall':    round(b['recall'],    4),
                'F1':        round(b['f1'],        4),
                'ROC-AUC':   round(b['roc_auc'],   4),
            })

df_clf = pd.DataFrame(clf_rows)
print("\n" + "="*20 + " BINARY CLASSIFICATION RESULTS " + "="*20)
display(df_clf.sort_values(by=['CV', 'Sensor', 'ROC-AUC'], ascending=[True, True, False]).round(4))

In [ ]:
# Tuned hyperparameters
for cv_name, cv_hp in all_cv_hyperparams.items():
    print(f"\n{'='*60}")
    print(f"  Tuned Hyperparameters \u2014 {cv_name}")
    print(f"{'='*60}")
    for sensor_name, sensor_hp in cv_hp.items():
        print(f"\n  {sensor_name}")
        for model_name in MODEL_NAMES:
            params = sensor_hp.get(model_name)
            if params is None:
                print(f"    {model_name}: GP \u2014 no Optuna parameters")
                continue
            print(f"    {model_name}:")
            for k, v in params.items():
                print(f"      {k}: {v}")

## Export Results

In [ ]:
results_bundle = {
    'cv_results':    all_cv_results,
    'hyperparams':   all_cv_hyperparams,
    'regression_df': df_reg,
    'clf_df':        df_clf,
    'sensor_config': SENSOR_CONFIG,
    'metadata': {
        'noise_level':       NOISE_LEVEL,
        'n_optuna_trials':   N_OPTUNA_TRIALS,
        'n_bootstrap':       N_BOOTSTRAP,
        'kfold_n_splits':    KFOLD_N_SPLITS,
        'source_file':       FILE_PATH,
        'prelim_file':       PRELIM_PATH,
        'model_names':       MODEL_NAMES,
        'cv_schemes':        list(CV_SCHEMES.keys()),
        'cv_scheme_details': (
            'LOGO = 5-fold (best-before date, cross-batch); '
            'KFold = 5-fold (random sweep-level split, within-distribution upper bound)'
        ),
    },
}

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results_bundle, f)

print(f"Full results saved to '{RESULTS_PATH}'")
print(f"  \u2192 Load with:  pd.read_pickle('{RESULTS_PATH}')")
print(f"  \u2192 CV schemes  : {list(CV_SCHEMES.keys())}")
print(f"  \u2192 Sensors     : {list(SENSOR_CONFIG.keys())}")
print(f"  \u2192 Models      : {MODEL_NAMES}")
print(f"  \u2192 Bootstrap   : {N_BOOTSTRAP} iterations")